In [1]:
import sys
sys.path.append("/kaggle/input/library-metrics")

In [2]:
import numpy as np
import pandas as pd
import random, time, psutil, threading, tracemalloc
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from metric_toolkit import (
    compute_energy, compute_carbon, compute_edp
)

# ============================================================
# 1️⃣ DỮ LIỆU
# ============================================================
X_train = pd.read_csv('/kaggle/input/data-time-complexity/X_train.csv')
y_train = pd.read_csv('/kaggle/input/data-time-complexity/y_train.csv').values.ravel()

X_train_fs, X_test_fs, y_train_fs, y_test_fs = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

# ============================================================
# 2️⃣ HÀM CHÍNH: GA + SVM
# ============================================================
def run_ga_fs(pop_size=20, num_generations=30, elite_size=2, mutation_rate=0.01):
    num_features = X_train_fs.shape[1]
    rng = np.random.default_rng(42)
    random.seed(42)

    def create_individual():
        position = rng.integers(0, 2, size=num_features).tolist()
        return {"position": position, "fitness": 0.0}

    num_scoring_evals = 0

    def fitness(position):
        nonlocal num_scoring_evals
        num_scoring_evals += 1
        mask = np.array(position, dtype=bool)
        if np.sum(mask) == 0:
            return 0.0
        X_tr_sel = X_train_fs.iloc[:, mask]
        X_te_sel = X_test_fs.iloc[:, mask]
        clf = SVC(C=0.1, kernel='linear', random_state=42)
        clf.fit(X_tr_sel, y_train_fs)
        score = clf.score(X_te_sel, y_test_fs)
        return float(score)

    def crossover(parent1, parent2):
        point = rng.integers(1, num_features - 1)
        child1 = parent1[:point] + parent2[point:]
        child2 = parent2[:point] + parent1[point:]
        return child1, child2

    def mutate(position):
        return [bit if random.random() > mutation_rate else 1 - bit for bit in position]

    population = [create_individual() for _ in range(pop_size)]
    gbest, gbest_score = population[0]["position"], 0.0

    for gen in range(num_generations):
        print(f"Generation {gen + 1}")
        for ind in population:
            score = fitness(ind["position"])
            ind["fitness"] = score
            if score > gbest_score:
                gbest = ind["position"][:]
                gbest_score = score

        elites = sorted(population, key=lambda x: x["fitness"], reverse=True)[:elite_size]
        new_population = elites[:]

        while len(new_population) < pop_size:
            parents = random.sample(population, 2)
            c1, c2 = crossover(parents[0]["position"], parents[1]["position"])
            c1 = mutate(c1)
            c2 = mutate(c2)
            new_population.append({"position": c1, "fitness": 0.0})
            if len(new_population) < pop_size:
                new_population.append({"position": c2, "fitness": 0.0})

        population = new_population[:pop_size]

    mask = np.array(gbest, dtype=bool)
    X_sel = X_train_fs.loc[:, mask]
    return mask, num_scoring_evals, X_sel

# ============================================================
# 3️⃣ LOGGER TRACEMALLOC (Python heap)
# ============================================================
mem_trace_log = []
logging_active = False

def log_tracemalloc(interval=0.1, max_samples=4000):
    start_time = time.time()
    count = 0
    while logging_active and count < max_samples:
        current, peak = tracemalloc.get_traced_memory()
        t = time.time() - start_time
        mem_trace_log.append((t, current / 1024**2, peak / 1024**2))
        time.sleep(interval)
        count += 1

# ============================================================
# 4️⃣ HÀM ĐO HIỆU NĂNG (CÓ TRACEMALLOC TIMELINE)
# ============================================================
def measure_time_and_memory_with_timeline(func, *args, **kwargs):
    proc = psutil.Process()
    tracemalloc.start()
    start_time = time.time()

    global logging_active
    logging_active = True
    logger_thread = threading.Thread(target=log_tracemalloc, args=(0.1,))
    logger_thread.start()

    result = func(*args, **kwargs)

    logging_active = False
    logger_thread.join()

    wall_time = time.time() - start_time
    current_mem, peak_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    cpu_util = psutil.cpu_percent(interval=None)
    peak_mem_mb = peak_mem / 1024**2

    return {
        "result": result,
        "WallTime(s)": wall_time,
        "CPUUtil_Avg(%)": cpu_util,
        "PeakMem(MB)": peak_mem_mb,
    }

# ============================================================
# 5️⃣ CHẠY GA + GHI LOG
# ============================================================
timing = measure_time_and_memory_with_timeline(run_ga_fs)
mask, num_evals, X_sel = timing["result"]

wall_time = timing["WallTime(s)"]
cpu_util  = timing["CPUUtil_Avg(%)"]
peak_mem  = timing["PeakMem(MB)"]
base_mem  = psutil.Process().memory_info().rss / 1024**2

# ============================================================
# 6️⃣ TÍNH METRICS NĂNG LƯỢNG
# ============================================================
energy = compute_energy(cpu_util, wall_time)
carbon = compute_carbon(energy)
edp    = compute_edp(energy, wall_time)

fs_metrics = {
    "FS_Method": "GA_SVM",
    "NumFeatures": int(mask.sum()),
    "NumEvals": num_evals,
    "WallTime(s)": wall_time,
    "CPUUtil(%)": cpu_util,
    "PeakMem(MB)": peak_mem,
    "BaseMem(MB)": base_mem,
    "Energy(J)": energy,
    "Carbon(gCO2e)": carbon,
    "EDP(J*s)": edp,
    "Timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
}

# ============================================================
# 7️⃣ LƯU FILE KẾT QUẢ
# ============================================================
pd.DataFrame([fs_metrics]).to_csv("/kaggle/working/fs_metrics_ga.csv", index=False)
np.save("/kaggle/working/ga_mask.npy", mask)

mem_df = pd.DataFrame(mem_trace_log, columns=["Time(s)", "Current(MB)", "Peak(MB)"])
mem_df.to_csv("/kaggle/working/tracemalloc_timeline_ga.csv", index=False)

# ============================================================
# 8️⃣ IN RA KẾT QUẢ
# ============================================================
print("✅ Genetic Algorithm Feature Selection done.")
print(f"→ Selected features: {mask.sum()}")
print(f"→ NumEvals: {num_evals}")
print(f"→ WallTime: {wall_time:.3f}s | CPU Avg: {cpu_util:.2f}%")
print(f"→ PeakMem (Python heap): {peak_mem:.2f} MB")
print(f"→ Carbon: {carbon:.4f} gCO2e")
print("→ Saved: fs_metrics_ga.csv, tracemalloc_timeline_ga.csv, ga_mask.npy")


Generation 1
Generation 2
Generation 3
Generation 4
Generation 5
Generation 6
Generation 7
Generation 8
Generation 9
Generation 10
Generation 11
Generation 12
Generation 13
Generation 14
Generation 15
Generation 16
Generation 17
Generation 18
Generation 19
Generation 20
Generation 21
Generation 22
Generation 23
Generation 24
Generation 25
Generation 26
Generation 27
Generation 28
Generation 29
Generation 30
✅ Genetic Algorithm Feature Selection done.
→ Selected features: 520
→ NumEvals: 600
→ WallTime: 24436.850s | CPU Avg: 25.60%
→ PeakMem (Python heap): 186.35 MB
→ Carbon: 21.7673 gCO2e
→ Saved: fs_metrics_ga.csv, tracemalloc_timeline_ga.csv, ga_mask.npy
